In [44]:
import os
from tqdm.auto import tqdm,trange
import pandas
import torch
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassF1Score
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torch.utils.data import Dataset
from torchvision.transforms import v2
import matplotlib.pyplot as plt
import torch.nn.functional as F
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
print(torch.cuda.get_device_name(device)
)

Using cuda device
AMD Radeon RX 7600S


In [41]:
target_transform = v2.Lambda(
    lambda y: F.one_hot(torch.tensor(y), num_classes=10).float()
)
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform
)

Trainig_data_ci = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform

)

test_data_ci = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform
)

In [42]:
class NeuralNetwork(nn.Module):
    def __init__(self, x = 0, y = 0, input = 28*28, first_hidden = 512,
                 second_hidden = 512, first_activation = nn.ReLU, second_activation = nn.ReLU):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_stack = nn.Sequential(
            nn.Linear(input, first_hidden),
            first_activation(),
            nn.Dropout(x),
            nn.Linear(first_hidden, second_hidden),
            second_activation(),
            nn.Dropout(y),
            nn.Linear(second_hidden, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_stack(x)
        return logits

In [49]:
history = {
    "train_loss": [], "train_acc": [], "train_prec": [], "train_f1": [],
    "val_loss": [],   "val_acc": [],   "val_prec": [],   "val_f1": [],
}
best_val_acc = 0.0
acc  = MulticlassAccuracy(num_classes=10).to(device)
prec = MulticlassPrecision(num_classes=10, average="macro").to(device)
f1   = MulticlassF1Score(num_classes=10, average="macro").to(device)
model = NeuralNetwork()
seed = 528
epoches = 15
loss_FM = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
train_split, val_split = random_split(training_data, [len(training_data)- int(len(training_data) * 0.2), int(len(training_data) * 0.2)],generator=torch.Generator().manual_seed(seed))
train_loader = DataLoader(dataset=train_split, batch_size=64, shuffle=True)
val_loader = DataLoader(dataset=val_split, batch_size=64, shuffle=True)

train_bar = tqdm(total=len(train_loader), desc="train")
val_bar   = tqdm(total=len(val_loader),   desc="val")

for epoch in trange(epoches, desc="Training"):

    train_loss_total = 0.0
    val_loss_total = 0.0
    model.train().to(device)

    train_bar.reset()
    train_bar.set_description(f"epoch {epoch} train")
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = loss_FM(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=1)
        acc_v = acc(preds, y)
        prec_v = prec(preds, y)
        f1_v = f1(preds, y)
        train_loss_total += loss.item()
        train_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc_v:.4f}",
                              prec=f"{prec_v:.4f}", f1=f"{f1_v:.4f}")
        train_bar.update(1)

    history["train_loss"].append(train_loss_total / len(train_loader))
    history["train_acc"].append(acc.compute().item())
    history["train_prec"].append(prec.compute().item())
    history["train_f1"].append(f1.compute().item())

    tqdm.write(f"epoch {epoch} train | loss {history['train_loss'][-1]:.4f} | "
                    f"Accuracy {history['train_acc'][-1]:.4f} | Precision {history['train_prec'][-1]:.4f} | "
                    f"F1 {history['train_f1'][-1]:.4f}")
    acc.reset()
    prec.reset()
    f1.reset()

    with torch.no_grad():
        model.eval()
        val_bar.reset()
        val_bar.set_description(f"epoch {epoch} val")
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = loss_FM(logits, y)
            preds = logits.argmax(dim=1)
            acc_v = acc(preds, y)
            prec_v = prec(preds, y)
            f1_v = f1(preds, y)
            val_loss_total += loss.item()
            val_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc_v:.4f}")
            val_bar.update(1)

    history["val_loss"].append(val_loss_total / len(val_loader))
    history["val_acc"].append(acc.compute().item())
    history["val_prec"].append(prec.compute().item())
    history["val_f1"].append(f1.compute().item())

    tqdm.write(f"epoch {epoch} val   | loss {history['val_loss'][-1]:.4f} | "
                    f"Accuracy {history['val_acc'][-1]:.4f} | Precision {history['val_prec'][-1]:.4f} | "
                    f"F1 {history['val_f1'][-1]:.4f}")
    if history["val_acc"][-1] > best_val_acc:
        best_val_acc = history["val_acc"][-1]
        torch.save(model.state_dict(), "best_model.pt")
        tqdm.write(f"new best val acc {best_val_acc:.4f}, saved")

    acc.reset()
    prec.reset()
    f1.reset()

train_bar.close()
val_bar.close()

train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5054 | Accuracy 0.8162 | Precision 0.8146 | F1 0.8142
epoch 0 val   | loss 0.4156 | Accuracy 0.8483 | Precision 0.8557 | F1 0.8476
new best val acc 0.8483, saved
epoch 1 train | loss 0.3893 | Accuracy 0.8586 | Precision 0.8582 | F1 0.8581
epoch 1 val   | loss 0.3830 | Accuracy 0.8621 | Precision 0.8635 | F1 0.8614
new best val acc 0.8621, saved
epoch 2 train | loss 0.3444 | Accuracy 0.8746 | Precision 0.8743 | F1 0.8741
epoch 2 val   | loss 0.3606 | Accuracy 0.8703 | Precision 0.8723 | F1 0.8699
new best val acc 0.8703, saved
epoch 3 train | loss 0.3219 | Accuracy 0.8822 | Precision 0.8819 | F1 0.8817
epoch 3 val   | loss 0.3350 | Accuracy 0.8758 | Precision 0.8757 | F1 0.8750
new best val acc 0.8758, saved
epoch 4 train | loss 0.3037 | Accuracy 0.8871 | Precision 0.8868 | F1 0.8867
epoch 4 val   | loss 0.3483 | Accuracy 0.8753 | Precision 0.8774 | F1 0.8755
epoch 5 train | loss 0.2902 | Accuracy 0.8901 | Precision 0.8896 | F1 0.8896
epoch 5 val   | loss 0.3735 |

In [50]:
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

test_model = NeuralNetwork()
test_model.load_state_dict(torch.load("best_model.pt", map_location=device))
test_model.to(device)
test_model.eval()

acc.reset(); prec.reset(); f1.reset()
test_loss_total = 0.0

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        logits = test_model(x)
        test_loss_total += loss_FM(logits, y).item()
        preds = logits.argmax(dim=1)
        acc.update(preds, y)
        prec.update(preds, y)
        f1.update(preds, y)

print(f"test | loss {test_loss_total/len(test_loader):.4f} | "
      f"Accuracy {acc.compute():.4f} | Precision {prec.compute():.4f} | F1 {f1.compute():.4f}")

test | loss 0.3844 | Accuracy 0.8800 | Precision 0.8811 | F1 0.8800
